In [ ]:
#| default_exp reconstruction

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch


def grid_splat(
    points,  # 3D coordinates in voxel space [0, D-1] x [0, H-1] x [0, W-1]
    values,  # Intensity values for each point - (N,)
    size: tuple[int, int, int],  # Dimensions of output volume - (D, H, W)
    mode: str = "bilinear",  # "nearest" or "bilinear" (trilinear in 3D)
):
    """Splat 3D points into a volume (inverse of grid_sample)."""
    if mode == "nearest":
        return splat_points_nearest(points, values, size)
    elif mode == "bilinear":
        return splat_points_trilinear(points, values, size)
    else:
        raise ValueError(f"Unsupported mode: {mode}. Use 'nearest' or 'bilinear'")


def splat_points_nearest(points, values, size):
    """Splat 3D points using nearest-neighbor interpolation."""
    device = points.device
    D, H, W = size

    # Initialize output
    volume = torch.zeros(D, H, W, device=device)
    counts = torch.zeros(D, H, W, device=device)

    # Round to nearest voxel
    i = torch.round(points[:, 0]).long()
    j = torch.round(points[:, 1]).long()
    k = torch.round(points[:, 2]).long()

    # Filter valid points (inside volume)
    valid = (i >= 0) & (i < D) & (j >= 0) & (j < H) & (k >= 0) & (k < W)

    if not valid.any():
        return volume

    # Apply filter
    i, j, k = i[valid], j[valid], k[valid]
    vals = values[valid]

    # Accumulate values
    volume.index_put_((i, j, k), vals, accumulate=True)
    counts.index_put_((i, j, k), torch.ones_like(vals), accumulate=True)

    # Normalize by count
    volume = volume / (counts + 1e-8)

    return volume


def splat_points_trilinear(points, values, size):
    """Splat 3D points using trilinear interpolation."""
    device = points.device
    D, H, W = size

    # Initialize output
    volume = torch.zeros(D, H, W, device=device)
    weights = torch.zeros(D, H, W, device=device)

    # Get integer voxel indices (floor)
    i0 = torch.floor(points[:, 0]).long()
    j0 = torch.floor(points[:, 1]).long()
    k0 = torch.floor(points[:, 2]).long()

    i1 = i0 + 1
    j1 = j0 + 1
    k1 = k0 + 1

    # Compute fractional parts for interpolation
    fi = points[:, 0] - i0.float()
    fj = points[:, 1] - j0.float()
    fk = points[:, 2] - k0.float()

    # Filter valid points (inside volume)
    valid = (i0 >= 0) & (i1 < D) & (j0 >= 0) & (j1 < H) & (k0 >= 0) & (k1 < W)

    if not valid.any():
        return volume

    # Apply filter
    i0, i1 = i0[valid], i1[valid]
    j0, j1 = j0[valid], j1[valid]
    k0, k1 = k0[valid], k1[valid]
    fi, fj, fk = fi[valid], fj[valid], fk[valid]
    vals = values[valid]

    # Compute 8 corner weights (trilinear interpolation weights)
    w000 = (1 - fi) * (1 - fj) * (1 - fk)
    w001 = (1 - fi) * (1 - fj) * fk
    w010 = (1 - fi) * fj * (1 - fk)
    w011 = (1 - fi) * fj * fk
    w100 = fi * (1 - fj) * (1 - fk)
    w101 = fi * (1 - fj) * fk
    w110 = fi * fj * (1 - fk)
    w111 = fi * fj * fk

    # Splat to 8 neighboring voxels
    volume.index_put_((i0, j0, k0), vals * w000, accumulate=True)
    volume.index_put_((i0, j0, k1), vals * w001, accumulate=True)
    volume.index_put_((i0, j1, k0), vals * w010, accumulate=True)
    volume.index_put_((i0, j1, k1), vals * w011, accumulate=True)
    volume.index_put_((i1, j0, k0), vals * w100, accumulate=True)
    volume.index_put_((i1, j0, k1), vals * w101, accumulate=True)
    volume.index_put_((i1, j1, k0), vals * w110, accumulate=True)
    volume.index_put_((i1, j1, k1), vals * w111, accumulate=True)

    # Accumulate weights for normalization
    weights.index_put_((i0, j0, k0), w000, accumulate=True)
    weights.index_put_((i0, j0, k1), w001, accumulate=True)
    weights.index_put_((i0, j1, k0), w010, accumulate=True)
    weights.index_put_((i0, j1, k1), w011, accumulate=True)
    weights.index_put_((i1, j0, k0), w100, accumulate=True)
    weights.index_put_((i1, j0, k1), w101, accumulate=True)
    weights.index_put_((i1, j1, k0), w110, accumulate=True)
    weights.index_put_((i1, j1, k1), w111, accumulate=True)

    # Normalize by total weight at each voxel
    volume = volume / (weights + 1e-8)

    return volume

In [ ]:
#| hide
import nbdev

nbdev.nbdev_export()